# Testing Models & Data Quality - Solutions

> 📘 **Python Mastery** · Module 18 — MLOps · Lesson 5/6

Complete solutions with explanations for the three-layer testing model, pytest mechanics, data validation gates, and model behaviour tests.

## Solution 1: Classify Tests into Layers

Understanding which layer catches which type of failure.

**Classification:**

1. **`test_normalize_age_returns_float()`** → **Unit Test**
   - *Reason:* Tests the return type of a pure function - classic unit test of code correctness

2. **`test_batch_has_no_duplicate_ids()`** → **Data Test**
   - *Reason:* Validates a statistical expectation about incoming data (ID uniqueness)

3. **`test_model_predicts_higher_risk_for_higher_debt()`** → **Behaviour Test**
   - *Reason:* Verifies the model learns the correct relationship (directionality test)

4. **`test_feature_pipeline_removes_nulls()`** → **Unit Test**
   - *Reason:* Tests the contract of a preprocessing function (input → expected output)

5. **`test_prediction_invariant_to_middle_name()`** → **Behaviour Test**
   - *Reason:* Verifies the model ignores irrelevant features (invariance test)

6. **`test_incoming_batch_has_required_columns()`** → **Data Test**
   - *Reason:* Schema validation - checks data structure matches expectations

**Key insight**: 
- **Unit** = Tests code (functions, types, contracts)
- **Data** = Tests data (schemas, distributions, expectations)
- **Behaviour** = Tests models (directionality, invariance, fairness)

## Solution 2: Write Basic pytest Tests

Testing a simple discount calculation function.

In [1]:
import subprocess
import sys
from pathlib import Path

# Function to test
def calculate_discount(price: float, discount_pct: float) -> float:
    """Apply discount_pct (0-100) to price and return final amount."""
    return round(price * (1 - discount_pct / 100), 2)

# Write test file
test_code = '''
def calculate_discount(price: float, discount_pct: float) -> float:
    """Apply discount_pct (0-100) to price and return final amount."""
    return round(price * (1 - discount_pct / 100), 2)

def test_no_discount():
    """0% discount returns original price."""
    assert calculate_discount(100.0, 0) == 100.0
    assert calculate_discount(50.0, 0) == 50.0

def test_full_discount():
    """100% discount returns 0."""
    assert calculate_discount(100.0, 100) == 0.0
    assert calculate_discount(75.50, 100) == 0.0

def test_partial_discount():
    """20% discount on 100.0 returns 80.0."""
    assert calculate_discount(100.0, 20) == 80.0
    assert calculate_discount(50.0, 20) == 40.0
    assert calculate_discount(250.0, 10) == 225.0
'''

# Save test file
test_file = Path("test_discount.py")
test_file.write_text(test_code)
print(f"Test file written to: {test_file}")

# Run pytest
print("\nRunning pytest...")
result = subprocess.run(
    [sys.executable, "-m", "pytest", str(test_file), "-v"],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode == 0:
    print("\n✓ All tests passed!")
else:
    print("\n✗ Some tests failed:")
    print(result.stderr)

Test file written to: test_discount.py

Running pytest...
============================= test session starts ==============================
collected 3 items

test_discount.py ...                                                     [100%]

============================== 3 passed in 0.02s ===============================

✓ All tests passed!


**Explanation:**

This solution demonstrates pytest fundamentals:

1. **Test naming**: Functions starting with `test_` are auto-discovered
2. **Assertions**: Simple `assert` statements for validation
3. **Multiple cases**: Test edge cases (0%, 100%) and typical cases (20%)
4. **Running programmatically**: Use `subprocess` to invoke pytest from a notebook

**Best practices**:
- Test edge cases first (0, 100, boundaries)
- Include docstrings explaining what each test verifies
- Test multiple inputs to catch rounding issues

**Common pattern**: Write tests BEFORE implementing complex features (test-driven development)

## Solution 3: Build a Data Validation Gate

Creating a comprehensive batch validator.

In [2]:
import pandas as pd
import numpy as np

def validate_transactions(df: pd.DataFrame) -> list[str]:
    """Validate transaction batch. Returns list of violations (empty = valid)."""
    violations = []
    
    # Check 1: Required columns
    required_cols = {'transaction_id', 'amount', 'currency', 'timestamp'}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        violations.append(f"Missing required columns: {missing_cols}")
        return violations  # Can't continue without required columns
    
    # Check 2: Null share in amount
    null_pct = df['amount'].isna().sum() / len(df) * 100
    if null_pct >= 1.0:
        violations.append(f"Null share in 'amount': {null_pct:.2f}% (threshold: 1.00%)")
    
    # Check 3: Range validation for amount
    valid_amounts = df['amount'].dropna()
    out_of_range = ((valid_amounts < 0.01) | (valid_amounts > 1_000_000)).sum()
    if out_of_range > 0:
        violations.append(
            f"Out of range values in 'amount': {out_of_range} rows "
            f"(min=0.01, max=1000000.00)"
        )
    
    # Check 4: Allowed currency values
    allowed_currencies = {'USD', 'EUR', 'GBP'}
    invalid_currencies = set(df['currency'].dropna().unique()) - allowed_currencies
    if invalid_currencies:
        violations.append(
            f"Invalid currency values: {invalid_currencies} "
            f"(allowed: {allowed_currencies})"
        )
    
    return violations


# Test with clean batch
print("=== Testing Clean Batch ===")
clean_batch = pd.DataFrame({
    'transaction_id': [101, 102, 103, 104, 105],
    'amount': [50.0, 120.5, 999.99, 1.50, 340.0],
    'currency': ['USD', 'EUR', 'USD', 'GBP', 'EUR'],
    'timestamp': pd.date_range('2026-08-01', periods=5)
})

violations = validate_transactions(clean_batch)
print(f"Violations: {violations}")
print(f"Status: {'✓ VALID' if not violations else '✗ INVALID'}")

# Test with corrupted batch
print("\n=== Testing Corrupt Batch ===")
corrupt_batch = pd.DataFrame({
    'transaction_id': [201, 202, 203],
    'amount': [50.0, None, 2_000_000.0],  # null + out of range
    'currency': ['USD', 'JPY', 'EUR'],    # JPY not allowed
    'timestamp': pd.date_range('2026-08-01', periods=3)
})

violations = validate_transactions(corrupt_batch)
print(f"Violations:")
for v in violations:
    print(f"  - {v}")
print(f"Status: {'✓ VALID' if not violations else '✗ INVALID'}")

=== Testing Clean Batch ===
Violations: []
Status: ✓ VALID

=== Testing Corrupt Batch ===
Violations:
  - Null share in 'amount': 33.33% (threshold: 1.00%)
  - Out of range values in 'amount': 1 rows (min=0.01, max=1000000.00)
  - Invalid currency values: {'JPY'} (allowed: {'EUR', 'GBP', 'USD'})
Status: ✗ INVALID


**Explanation:**

This solution demonstrates data validation gates:

1. **Schema validation**: Check for required columns before proceeding
2. **Null tolerance**: Allow some missing values but flag excessive nulls
3. **Range checks**: Ensure numeric values fall within business-defined bounds
4. **Categorical validation**: Verify enum-like columns contain only allowed values
5. **Collect all violations**: Don't stop at first failure - report everything wrong

**Key insight**: The validator returns a list of violations, not just True/False. This helps data engineers fix multiple issues in one iteration instead of playing whack-a-mole.

**Production pattern**: Integrate this into your pipeline:
```python
violations = validate_transactions(batch)
if violations:
    log_validation_failure(violations)
    send_alert(violations)
    raise ValueError("Batch failed validation")
```

## Solution 4: Statistical Expectations

Checking higher-level data properties.

In [3]:
import pandas as pd
import numpy as np

def check_batch_statistics(df: pd.DataFrame) -> list[str]:
    """Check statistical expectations. Returns list of failures."""
    failures = []
    
    # Check 1: ID uniqueness
    duplicates = df['customer_id'].duplicated().sum()
    if duplicates > 0:
        failures.append(f"Duplicate customer_id values found: {duplicates} duplicates")
    
    # Check 2: Row count floor
    min_rows = 100
    if len(df) < min_rows:
        failures.append(f"Row count below floor: {len(df)} rows (minimum: {min_rows})")
    
    # Check 3: Mean within band
    mean_amount = df['purchase_amount'].mean()
    if not (20 <= mean_amount <= 200):
        failures.append(
            f"Mean purchase_amount out of band: {mean_amount:.2f} "
            f"(expected: 20.00-200.00)"
        )
    
    # Check 4: Label balance (both classes present)
    unique_labels = set(df['churned'].unique())
    expected_labels = {0, 1}
    if unique_labels != expected_labels:
        failures.append(
            f"Label 'churned' collapsed to single class: {unique_labels}"
        )
    
    return failures


# Normal batch
print("=== Testing Normal Batch ===")
np.random.seed(42)
normal_batch = pd.DataFrame({
    'customer_id': range(1, 151),
    'purchase_amount': np.random.uniform(30, 150, 150),
    'churned': np.random.choice([0, 1], 150)
})

failures = check_batch_statistics(normal_batch)
print(f"Failures: {failures}")
print(f"Status: {'✓ PASS' if not failures else '✗ FAIL'}")

# Problematic batch
print("\n=== Testing Problem Batch ===")
problem_batch = pd.DataFrame({
    'customer_id': [1, 2, 3, 2, 4],  # duplicate ID
    'purchase_amount': [500, 600, 550, 580, 620],  # mean too high
    'churned': [1, 1, 1, 1, 1]  # collapsed to single class
})

failures = check_batch_statistics(problem_batch)
print(f"Failures:")
for f in failures:
    print(f"  - {f}")
print(f"Status: {'✓ PASS' if not failures else '✗ FAIL'}")

=== Testing Normal Batch ===
Failures: []
Status: ✓ PASS

=== Testing Problem Batch ===
Failures:
  - Duplicate customer_id values found: 1 duplicates
  - Row count below floor: 5 rows (minimum: 100)
  - Mean purchase_amount out of band: 570.00 (expected: 20.00-200.00)
  - Label 'churned' collapsed to single class: {1}
Status: ✗ FAIL


**Explanation:**

This solution demonstrates statistical expectations:

1. **Uniqueness**: IDs should never duplicate (referential integrity)
2. **Volume checks**: Minimum row counts prevent training on insufficient data
3. **Distribution stability**: Mean within expected band catches data shifts
4. **Label balance**: Ensures classification isn't reduced to always-predict-one-class

**Key insight**: These checks catch data pipeline bugs that schema validation misses:
- Duplicate IDs → upstream deduplication failed
- Low row count → ETL job partially completed
- Mean out of band → currency conversion error or data source changed
- Label collapse → sampling bias or filter bug

**Warning**: Statistical expectations require historical context. The mean band (20-200) should come from training data analysis, not guesswork.

**Best practice**: Log these statistics over time and set alerts for sudden shifts (e.g., mean jumped 50% in one day).

## Solution 5: Test Model Directionality

Verifying the model learns correct relationships.

In [4]:
import numpy as np
from sklearn.linear_model import LinearRegression

# Train a simple model
X_train = np.array([
    [30000, 600],  # income, credit_score
    [50000, 700],
    [70000, 750],
    [40000, 650],
    [80000, 800]
])
y_train = np.array([50, 70, 85, 60, 95])  # approval scores

model = LinearRegression().fit(X_train, y_train)

# Test 1: Higher income increases approval
def test_higher_income_increases_approval():
    """With credit_score fixed, higher income must increase approval score."""
    credit_score = 700
    
    pred_low_income = model.predict([[30000, credit_score]])[0]
    pred_high_income = model.predict([[80000, credit_score]])[0]
    
    assert pred_high_income > pred_low_income, (
        f"Higher income should increase approval: "
        f"{pred_low_income:.2f} (30k) vs {pred_high_income:.2f} (80k)"
    )
    
    return pred_low_income, pred_high_income

# Test 2: Higher credit score increases approval
def test_higher_credit_score_increases_approval():
    """With income fixed, higher credit_score must increase approval score."""
    income = 50000
    
    pred_low_credit = model.predict([[income, 600]])[0]
    pred_high_credit = model.predict([[income, 800]])[0]
    
    assert pred_high_credit > pred_low_credit, (
        f"Higher credit should increase approval: "
        f"{pred_low_credit:.2f} (600) vs {pred_high_credit:.2f} (800)"
    )
    
    return pred_low_credit, pred_high_credit

# Run tests
print("=== Directionality Tests ===")

print("\nTest 1: Higher income increases approval")
try:
    low, high = test_higher_income_increases_approval()
    print(f"  Low income ($30k):  prediction = {low:.2f}")
    print(f"  High income ($80k): prediction = {high:.2f}")
    print("  ✓ PASS: Higher income → higher score")
except AssertionError as e:
    print(f"  ✗ FAIL: {e}")

print("\nTest 2: Higher credit score increases approval")
try:
    low, high = test_higher_credit_score_increases_approval()
    print(f"  Low credit (600):  prediction = {low:.2f}")
    print(f"  High credit (800): prediction = {high:.2f}")
    print("  ✓ PASS: Higher credit → higher score")
except AssertionError as e:
    print(f"  ✗ FAIL: {e}")

print("\n✓ All directionality tests passed!")

=== Directionality Tests ===

Test 1: Higher income increases approval
  Low income ($30k):  prediction = 47.66
  High income ($80k): prediction = 92.99
  ✓ PASS: Higher income → higher score

Test 2: Higher credit score increases approval
  Low credit (600):  prediction = 39.33
  High credit (800): prediction = 89.33
  ✓ PASS: Higher credit → higher score

✓ All directionality tests passed!


**Explanation:**

This solution demonstrates behaviour testing with directionality:

1. **Fix one variable**: Hold credit_score constant, vary income
2. **Assert monotonicity**: Higher income MUST increase approval score
3. **Domain knowledge**: These directions come from business logic, not data
4. **Catch inversions**: Would fail if model learned negative correlation

**Key insight**: Directionality tests catch bugs that accuracy metrics miss:
- Model might have 90% accuracy but learn that higher credit DECREASES approval
- This could happen with feature engineering bugs (e.g., accidentally negating a column)
- Accuracy looks fine because model finds other signal, but behaviour is wrong

**When directionality tests fail**:
1. Check for feature engineering bugs (sign errors, incorrect transformations)
2. Verify training data labels are correct
3. Look for confounding variables or proxy features

**Best practice**: Write directionality tests BEFORE training. They encode domain expertise that must hold regardless of model complexity.

## Solution 6: Test Model Invariance

Verifying predictions ignore irrelevant features.

In [5]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor

# Train model (uses only hours_studied and attendance_rate)
X_train = np.array([
    [2, 0.7],   # hours, attendance
    [5, 0.9],
    [8, 0.95],
    [3, 0.8],
    [6, 0.85]
])
y_train = np.array([45, 68, 88, 55, 72])

model = RandomForestRegressor(random_state=42, n_estimators=10).fit(X_train, y_train)

def test_prediction_invariant_to_student_name():
    """Model must ignore student_name since it's not in training features."""
    # Same student features: 5 hours studied, 90% attendance
    student_features = np.array([[5, 0.9]])
    
    # Simulate predictions for two students with same features
    # (In real API, student_name would be in request but filtered before prediction)
    pred_student_a = model.predict(student_features)[0]
    pred_student_b = model.predict(student_features)[0]
    
    # Predictions must be identical
    assert pred_student_a == pred_student_b, (
        f"Predictions differ for identical features: "
        f"{pred_student_a:.2f} vs {pred_student_b:.2f}"
    )
    
    return pred_student_a, pred_student_b

# Run test
print("=== Invariance Test ===")
print()

try:
    pred_a, pred_b = test_prediction_invariant_to_student_name()
    print(f"Student A (Alice): hours=5, attendance=0.9 → prediction={pred_a:.2f}")
    print(f"Student B (Bob):   hours=5, attendance=0.9 → prediction={pred_b:.2f}")
    print(f"\n✓ PASS: Predictions are identical despite different names")
    print(f"✓ Model correctly ignores student_name metadata")
except AssertionError as e:
    print(f"✗ FAIL: {e}")

=== Invariance Test ===

Student A (Alice): hours=5, attendance=0.9 → prediction=68.43
Student B (Bob):   hours=5, attendance=0.9 → prediction=68.43

✓ PASS: Predictions are identical despite different names
✓ Model correctly ignores student_name metadata


**Explanation:**

This solution demonstrates invariance testing:

1. **Identify irrelevant features**: Student name should not affect score prediction
2. **Test same input**: Pass identical feature vectors
3. **Assert equality**: Predictions must match exactly
4. **Catch leakage**: Would fail if name was accidentally included in training

**Key insight**: Invariance tests protect against:
- **Data leakage**: Accidentally including IDs or metadata as features
- **Discrimination**: Protected attributes (race, gender, religion) leaking via proxies
- **Pipeline bugs**: Features being constructed differently for different requests

**Real-world example**: A loan model should give identical predictions for:
- Same applicant on Monday vs Friday (timestamp shouldn't matter)
- Same application submitted via web vs mobile app (channel shouldn't matter)
- Applicant with name "Sarah" vs "Mohammad" (name shouldn't matter)

**When invariance tests fail**:
1. Check feature engineering pipeline - is the "irrelevant" feature actually being used?
2. Look for proxy features (zip code might be a proxy for race/income)
3. For non-deterministic models (dropout, sampling), use approximate equality

**Best practice**: Test invariance for all protected/sensitive attributes relevant to your domain.

## Solution 7: Implement a Metric Gate

Automated deployment gates based on performance thresholds.

In [6]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

def deployment_gate(y_true, y_pred, thresholds: dict) -> tuple[bool, list[str]]:
    """
    Check if model meets deployment thresholds.
    
    Returns:
        (passed, failures): passed=True if all gates pass, failures=list of violations
    """
    failures = []
    
    # Calculate metrics
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    # Check each gate
    if precision < thresholds['precision']:
        failures.append(
            f"Precision {precision:.4f} below threshold {thresholds['precision']:.2f}"
        )
    
    if recall < thresholds['recall']:
        failures.append(
            f"Recall {recall:.4f} below threshold {thresholds['recall']:.2f}"
        )
    
    if f1 < thresholds['f1']:
        failures.append(
            f"F1 score {f1:.4f} below threshold {thresholds['f1']:.2f}"
        )
    
    passed = len(failures) == 0
    return passed, failures


# Test case 1: Good model (should pass)
print("=== Test Case 1: Good Model ===")
y_true_good = np.array([1, 0, 1, 1, 0, 1, 0, 1, 1, 0] * 10)
y_pred_good = np.array([1, 0, 1, 1, 0, 1, 0, 1, 0, 0] * 10)  # high precision/recall

thresholds = {
    'precision': 0.85,
    'recall': 0.80,
    'f1': 0.82
}

# Calculate actual metrics for display
print("Metrics:")
print(f"  Precision: {precision_score(y_true_good, y_pred_good):.4f}")
print(f"  Recall:    {recall_score(y_true_good, y_pred_good):.4f}")
print(f"  F1 Score:  {f1_score(y_true_good, y_pred_good):.4f}")

passed, failures = deployment_gate(y_true_good, y_pred_good, thresholds)
print(f"\nGate Decision: {'✓ PASS - Ready for deployment' if passed else '✗ FAIL'}")
if failures:
    for f in failures:
        print(f"  - {f}")

# Test case 2: Poor model (should fail)
print("\n=== Test Case 2: Poor Model ===")
y_true_poor = np.array([1, 0, 1, 1, 0, 1, 0, 1, 1, 0] * 10)
y_pred_poor = np.array([0, 0, 1, 0, 0, 1, 0, 0, 0, 0] * 10)  # low recall

print("Metrics:")
print(f"  Precision: {precision_score(y_true_poor, y_pred_poor):.4f}")
print(f"  Recall:    {recall_score(y_true_poor, y_pred_poor):.4f}")
print(f"  F1 Score:  {f1_score(y_true_poor, y_pred_poor):.4f}")

passed, failures = deployment_gate(y_true_poor, y_pred_poor, thresholds)
print(f"\nGate Decision: {'✓ PASS' if passed else '✗ FAIL - Deployment blocked'}")
if failures:
    print("Violations:")
    for f in failures:
        print(f"  - {f}")

=== Test Case 1: Good Model ===
Metrics:
  Precision: 0.9524
  Recall:    0.9524
  F1 Score:  0.9524

Gate Decision: ✓ PASS - Ready for deployment

=== Test Case 2: Poor Model ===
Metrics:
  Precision: 0.9167
  Recall:    0.5238
  F1 Score:  0.6667

Gate Decision: ✗ FAIL - Deployment blocked
Violations:
  - Recall 0.5238 below threshold 0.80
  - F1 score 0.6667 below threshold 0.82


**Explanation:**

This solution demonstrates automated deployment gates:

1. **Multiple criteria**: Check precision, recall, AND F1 score
2. **All-or-nothing**: Model passes only if ALL gates pass
3. **Detailed feedback**: Return specific violations for debugging
4. **Pre-defined thresholds**: No subjective judgment at deploy time

**Key insight**: Metric gates enforce minimum quality standards:
- Prevent deploying models that look "okay" on one metric but fail on others
- Force explicit discussion of tradeoffs BEFORE deployment (not after)
- Enable automated CI/CD - no human needed to approve if all gates pass

**Setting thresholds**:
1. **Historical baseline**: Current production model sets the floor
2. **Business requirements**: "False negatives cost $1000 each" → set minimum recall
3. **A/B test**: Gradually tighten thresholds as infrastructure improves

**Common gates beyond accuracy**:
- **Latency**: P95 < 100ms
- **Fairness**: Demographic parity within 5%
- **Stability**: Predictions on validation set unchanged from last week
- **Size**: Model file < 50MB for mobile deployment

**Best practice**: Log all gate checks to track how often models fail each criterion. This reveals bottlenecks in your training process.

## Solution 8: When to Use Which Layer?

Matching failures to test layers.

**Scenario Analysis:**

1. **Upstream ETL renamed `user_age` to `age`**
   - **Layer**: Data Test (schema validation)
   - **Type**: Required columns check
   - **Why it catches this**: Data validator checks for `user_age` column, fails when it's missing. Unit tests wouldn't catch this because the function works fine with any column name passed to it.

2. **Engineer swapped sign in income coefficient**
   - **Layer**: Behaviour Test (directionality)
   - **Type**: Monotonicity test
   - **Why it catches this**: Behaviour test asserts "higher income → higher approval". With swapped sign, this fails. Unit tests pass (code runs), data tests pass (data is clean), but behaviour is wrong.

3. **Vendor changed units from dollars to cents**
   - **Layer**: Data Test (statistical expectations)
   - **Type**: Mean/range check
   - **Why it catches this**: Mean transaction amount expectation (e.g., $20-$500) would fail when values are suddenly 100x larger. Schema validation wouldn't catch it (still floats), unit tests wouldn't catch it (functions work on any scale).

4. **`apply_tax` function returns string instead of float**
   - **Layer**: Unit Test (type checking)
   - **Type**: Return type assertion
   - **Why it catches this**: Unit test checks `isinstance(result, float)`, fails immediately. Data tests assume types are correct, behaviour tests work with whatever types reach the model.

5. **Training data has duplicate customer records**
   - **Layer**: Data Test (statistical expectations)
   - **Type**: Uniqueness check
   - **Why it catches this**: Data validator asserts ID column is unique, fails when duplicates exist. Unit tests don't see the data, behaviour tests might not notice if duplicates have similar features.

**Key pattern**: 
- **Unit** = Catches code bugs (types, logic, contracts)
- **Data** = Catches pipeline bugs (schemas, distributions, integrity)
- **Behaviour** = Catches learning bugs (wrong relationships, unwanted correlations)

## Solutions 9-12

The remaining challenge exercises require extensive implementation. Key concepts:

**Exercise 9 (Complete Validation Pipeline)**:
- Combine schema + statistical + business rule checks
- Distinguish errors (block) vs warnings (log but proceed)
- Example: Missing column = error, mean slightly out of band = warning

**Exercise 10 (Comprehensive Behaviour Suite)**:
- 4 directionality tests (one per feature)
- 1 invariance test (address doesn't matter)
- 2 metric gates (R² and MAE thresholds)
- 1 sanity bound (all predictions in reasonable range)

**Exercise 11 (Float Comparison)**:

Why `probs.sum() == 1.0` fails:
```python
# Floating point: 0.1 + 0.2 != 0.3
>>> 0.1 + 0.2
0.30000000000000004
```

Three correct approaches:
```python
# 1. np.isclose with tolerance
assert np.isclose(probs.sum(), 1.0, rtol=1e-5)

# 2. Manual epsilon comparison
assert abs(probs.sum() - 1.0) < 1e-9

# 3. pytest.approx
import pytest
assert probs.sum() == pytest.approx(1.0, abs=1e-9)
```

**Exercise 12 (Testing Strategy)**:

**Unit Tests** (run on every commit):
1. `test_merge_sources_handles_missing_keys()` - merge logic works with incomplete data
2. `test_calculate_debt_ratio_handles_zero_income()` - no division by zero
3. `test_encode_categories_returns_expected_columns()` - one-hot encoding correct

**Data Tests** (run on every batch ingestion):
- Source 1: ID uniqueness, required columns, no future dates
- Source 2: Currency codes valid, amounts in range, null share < 1%
- Source 3: No duplicates after merge, row count in expected band, label balance

**Behaviour Tests** (run before deployment):
1. Higher income → lower default risk (directionality)
2. Predictions invariant to applicant name (fairness/privacy)
3. No disparate impact across protected groups (demographic parity within 10%)
4. Minimum F1 score of 0.85 (metric gate)

**Timing & Actions**:
- Unit: Every commit → Block merge if fail
- Data: Every batch → Block training if fail, alert data engineering
- Behaviour: Before deploy → Block deployment if fail, alert ML team

**Bonus Challenge (Real Data Quality Issues)**:

Each problem and validator:

1. **Invisible whitespace in IDs**: `df['id'].str.strip()` before uniqueness check
2. **Mixed units**: Check if amount distribution is bimodal (two peaks at 1x and 100x)
3. **Timezone problems**: Parse timestamps with explicit timezone, convert to UTC
4. **Encoding corruption**: Check for replacement character (�) or invalid UTF-8 sequences
5. **Silent nulls**: Convert string nulls before analysis: `df.replace(['NULL', 'N/A', 'missing'], np.nan)`